Import packages

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

Data

In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

Functions

In [6]:
class ConfigurableCNN(nn.Module):
    def __init__(self, activation='relu', pooling='max', extra_layer=False):
        super().__init__()

        if activation == 'relu':
            act = nn.ReLU
        elif activation == 'sigmoid':
            act = nn.Sigmoid
        else:
            raise ValueError("activation must be 'relu' or 'sigmoid'")

        if pooling == 'max':
            pool = nn.MaxPool2d
        elif pooling == 'avg':
            pool = nn.AvgPool2d
        else:
            raise ValueError("pooling must be 'max' or 'avg'")

        layers = [
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            act(),
            pool(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            act(),
            pool(2)
        ]

        if extra_layer:
            layers.extend([
                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                act()
            ])
            self.features = nn.Sequential(*layers)
            self.classifier = nn.Linear(64 * 8 * 8, 10)
        else:
            self.features = nn.Sequential(*layers)
            self.classifier = nn.Linear(32 * 8 * 8, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

def train_model(model, optimizer, epochs=2):
    criterion = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(trainloader):.4f}")

def test_model(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

def run_experiment(name, activation='relu', pooling='max', optimizer_name='adam', extra_layer=False, lr=None):
    model = ConfigurableCNN(
        activation=activation,
        pooling=pooling,
        extra_layer=extra_layer
    ).to(device)

    if optimizer_name == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=0.001 if lr is None else lr)
    elif optimizer_name == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=0.01 if lr is None else lr)
    else:
        raise ValueError("optimizer_name must be 'adam' or 'sgd'")

    train_model(model, optimizer, epochs=2)
    acc = test_model(model)
    print(f"{name}: {acc:.2f}%")
    return acc

Results

In [9]:
results = {}

# 1. Optimizer comparison
results["Adam / ReLU / MaxPool / 2-layer"] = run_experiment(
    "Adam / ReLU / MaxPool / 2-layer",
    activation='relu',
    pooling='max',
    optimizer_name='adam',
    extra_layer=False
)

results["SGD / ReLU / MaxPool / 2-layer"] = run_experiment(
    "SGD / ReLU / MaxPool / 2-layer",
    activation='relu',
    pooling='max',
    optimizer_name='sgd',
    extra_layer=False
)

# 2. Activation comparison
results["Adam / Sigmoid / MaxPool / 2-layer"] = run_experiment(
    "Adam / Sigmoid / MaxPool / 2-layer",
    activation='sigmoid',
    pooling='max',
    optimizer_name='adam',
    extra_layer=False
)

# 3. Pooling comparison
results["Adam / ReLU / AvgPool / 2-layer"] = run_experiment(
    "Adam / ReLU / AvgPool / 2-layer",
    activation='relu',
    pooling='avg',
    optimizer_name='adam',
    extra_layer=False
)

# 4. Depth comparison
results["Adam / ReLU / MaxPool / 3-layer"] = run_experiment(
    "Adam / ReLU / MaxPool / 3-layer",
    activation='relu',
    pooling='max',
    optimizer_name='adam',
    extra_layer=True
)

print("\nFinal Results:")
for k, v in results.items():
    print(f"{k}: {v:.2f}%")

Epoch 1/2, Loss: 1.4955
Epoch 2/2, Loss: 1.1866
Adam / ReLU / MaxPool / 2-layer: 59.48%
Epoch 1/2, Loss: 1.9568
Epoch 2/2, Loss: 1.6398
SGD / ReLU / MaxPool / 2-layer: 41.12%
Epoch 1/2, Loss: 1.9627
Epoch 2/2, Loss: 1.6884
Adam / Sigmoid / MaxPool / 2-layer: 43.82%
Epoch 1/2, Loss: 1.5492
Epoch 2/2, Loss: 1.2539
Adam / ReLU / AvgPool / 2-layer: 57.01%
Epoch 1/2, Loss: 1.4406
Epoch 2/2, Loss: 1.0991
Adam / ReLU / MaxPool / 3-layer: 63.00%

Final Results:
Adam / ReLU / MaxPool / 2-layer: 59.48%
SGD / ReLU / MaxPool / 2-layer: 41.12%
Adam / Sigmoid / MaxPool / 2-layer: 43.82%
Adam / ReLU / AvgPool / 2-layer: 57.01%
Adam / ReLU / MaxPool / 3-layer: 63.00%


Note: I reran this code after obtaining the initial results, which gave me slightly different results. So, the numbers shown here are slightly different than the ones shown in my report paper. However, the overall trends have not changed.